# 03 — Supervised Fine-Tuning (SFT) with Qwen

Fine-tune a Qwen2.5 model on the DAIC-WOZ clinical interview data using LoRA + TRL's `SFTTrainer`.

The dataset (`daicwoz_finetune.jsonl`) is already in chat format — each line is a full interview formatted as `{"messages": [{role, content}, ...]}`.


In [1]:
import json
import os
from pathlib import Path

import torch
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


/home/kevin/cse-reu/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.12.0+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
VRAM: 8.5 GB


## Configuration

In [2]:
# ── Model ──────────────────────────────────────────────────────────────────
# Options: "Qwen/Qwen2.5-0.5B-Instruct"  (lightest, good for testing)
#          "Qwen/Qwen2.5-1.5B-Instruct"
#          "Qwen/Qwen2.5-7B-Instruct"    (needs ~16 GB VRAM without quant)
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_PATH  = Path("../data/processed/daicwoz_finetune.jsonl")
OUTPUT_DIR = Path("../models/qwen2.5-ellie-sft")

# ── LoRA ───────────────────────────────────────────────────────────────────
LORA_R         = 16
LORA_ALPHA     = 32
LORA_DROPOUT   = 0.05

# ── Training ───────────────────────────────────────────────────────────────
MAX_SEQ_LENGTH = 2048
NUM_EPOCHS     = 3
BATCH_SIZE     = 2        # per-device; lower if OOM
GRAD_ACCUM     = 4        # effective batch = BATCH_SIZE * GRAD_ACCUM
LR             = 2e-4

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Model : {MODEL_NAME}")
print(f"Output: {OUTPUT_DIR}")


Model : Qwen/Qwen2.5-0.5B-Instruct
Output: ../models/qwen2.5-ellie-sft


## Load Dataset

In [3]:
records = []
with open(DATA_PATH) as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

dataset = Dataset.from_list(records)

print(f"Conversations : {len(dataset)}")
print(f"Columns       : {dataset.column_names}")
print(f"\nExample (first 2 turns):")
for msg in dataset[0]["messages"][:2]:
    print(f"  [{msg['role']}]: {msg['content'][:120]!r}")


Conversations : 186
Columns       : ['messages']

Example (first 2 turns):
  [system]: 'You are Ellie, a virtual clinical interviewer. Conduct a structured, empathetic mental health interview by asking open-e'
  [assistant]: "Hi, I'm Ellie. Thanks for coming in today. I was created to talk to people in a safe and secure environment. Think of me"


## Load Tokenizer & Model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"   # required for causal LM training

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False
# Explicitly align pad/eos token IDs so transformers doesn't auto-correct and warn
model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id
if hasattr(model, "generation_config"):
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    model.generation_config.eos_token_id = tokenizer.eos_token_id

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Parameters  : {total:,}  (trainable before LoRA: {trainable:,})")


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 924.34it/s]


Parameters  : 494,032,768  (trainable before LoRA: 494,032,768)


## Apply LoRA

In [5]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    # Qwen2.5 attention + MLP projection layers
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


## Train

In [7]:
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

sft_config = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    max_length=MAX_SEQ_LENGTH,       # renamed from max_seq_length in TRL 1.x
    # Logging & saving
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    # Precision
    bf16=use_bf16,
    fp16=(not use_bf16) and torch.cuda.is_available(),
    # Misc
    gradient_checkpointing=True,
    report_to="none",
    dataset_kwargs={"skip_prepare_dataset": False},
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print("Trainer ready.")
print(f"Steps per epoch : {len(trainer.get_train_dataloader())}")
print(f"Total steps     : {sft_config.max_steps if sft_config.max_steps > 0 else 'auto'}")


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Tokenizing train dataset: 100%|██████████| 186/186 [00:02<00:00, 85.02 examples/s]

Trainer ready.
Steps per epoch : 93
Total steps     : auto


In [8]:
train_result = trainer.train()

print("\n── Training complete ──")
print(f"Runtime    : {train_result.metrics['train_runtime']:.1f}s")
print(f"Samples/s  : {train_result.metrics['train_samples_per_second']:.2f}")
print(f"Final loss : {train_result.metrics['train_loss']:.4f}")


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss


KeyboardInterrupt: 

## Save Model (LoRA adapters + merged)

In [ ]:
# Save LoRA adapters (lightweight, ~few MB)
adapter_dir = OUTPUT_DIR / "lora-adapters"
model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
print(f"LoRA adapters saved → {adapter_dir}")

# Optionally merge LoRA weights into base model and save full weights
# Uncomment to get a stand-alone model (larger, ~1 GB for 0.5B)
# merged = model.merge_and_unload()
# merged_dir = OUTPUT_DIR / "merged"
# merged.save_pretrained(str(merged_dir), safe_serialization=True)
# tokenizer.save_pretrained(str(merged_dir))
# print(f"Merged model saved → {merged_dir}")


## Quick Inference Test

In [ ]:
model.eval()

test_messages = [
    {"role": "system",    "content": "You are Ellie, a virtual clinical interviewer. Conduct a structured, empathetic mental health interview by asking open-ended questions and responds naturally to what the participant shares."},
    {"role": "assistant", "content": "Hi, I'm Ellie. How are you doing today?"},
    {"role": "user",      "content": "I've been feeling pretty down lately."},
]

input_ids = tokenizer.apply_chat_template(
    test_messages,
    add_generation_prompt=True,
    return_tensors="pt",
).to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        input_ids,
        max_new_tokens=128,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )

new_tokens = output_ids[0][input_ids.shape[-1]:]
response = tokenizer.decode(new_tokens, skip_special_tokens=True)
print("Ellie:", response)
